# Dioptra-DINO: Foundation-Assisted Metric Depth on Edge Devices

This notebook trains **Dioptra-DINO** (~25.4M parameters) on dual NVIDIA T4 GPUs on Kaggle.

### Architecture Highlights:
- **Pre-trained Visual Backbone**: DINOv2-Small (`vits14`, 21.6M parameters pre-trained on 142M images).
- **Trivision Ray Positional Encoding**: Continuous optical ray unprojection from camera matrix $\mathbf{K}$.
- **Angular Residual Attention (ARA)**: Intrinsic geometric attention bias $\sin^2(\theta_{q, k})$ enforcing surface planarity.
- **Decoupled Scale Supervision**: Log-median metric scale loss $\mathcal{L}_{\text{scale}}$ ensuring accurate physical ranging in metres.

### Kaggle Setup Instructions:
1. **Settings (Right Sidebar)**:
   - **Accelerator**: GPU T4 x2 (or P100)
   - **Internet**: **ON** (allows automatic download of DINOv2 weights on first cell)
2. **+ Add Input**:
   - Attach dataset: **`pandrii000/dasvo-tartanair-rgb-d-validation-split`**
   - Attach code repository (contains `dioptra_dino.py`)

In [ ]:
# [2] Locate and update dioptra_dino.py (supports Kaggle input upload OR auto-cloning/pulling from GitHub)
import glob, os, shutil, sys

repo_dir = "/kaggle/working/dioptra_repo"
if os.path.exists(repo_dir):
    print("Updating existing dioptra_repo from GitHub...")
    os.system(f"cd {repo_dir} && git pull")
else:
    print("Cloning dioptra repository from GitHub...")
    os.system(f"git clone https://github.com/SeranomTheGreat/dioptra.git {repo_dir}")

for fname in ["dioptra_dino.py", "dioptra.py"]:
    src = os.path.join(repo_dir, fname)
    dst = os.path.join("/kaggle/working", fname)
    if os.path.exists(src):
        shutil.copyfile(src, dst)
        print(f"Synchronized {fname} to /kaggle/working/")

for p in ["/kaggle/working", repo_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)
print("Environment initialized!")


In [ ]:
# [2] Locate dioptra_dino.py (supports Kaggle input upload OR auto-cloning from GitHub)
import glob, os, shutil

candidates = sorted(glob.glob("/kaggle/input/**/dioptra_dino.py", recursive=True))
if not candidates:
    if os.path.exists("dioptra_dino.py"):
        candidates = ["dioptra_dino.py"]
    elif os.path.exists("../dioptra_dino.py"):
        candidates = ["../dioptra_dino.py"]
    else:
        print("dioptra_dino.py not mounted in /kaggle/input. Auto-cloning from GitHub...")
        os.system("git clone https://github.com/SeranomTheGreat/dioptra.git /kaggle/working/dioptra_repo")
        git_src = "/kaggle/working/dioptra_repo/dioptra_dino.py"
        if os.path.exists(git_src):
            candidates = [git_src]

assert candidates, "Could not locate dioptra_dino.py! Please ensure Internet is ON or upload dioptra_dino.py."
script_src = candidates[-1]
script_dst = "/kaggle/working/dioptra_dino.py"
if os.path.abspath(script_src) != os.path.abspath(script_dst):
    shutil.copyfile(script_src, script_dst)
print(f"Ready! Using script at: {script_dst}")


In [ ]:
# [4] Discover TartanAir dataset path
import os, sys
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

from dioptra_dino import resolve_dataset_root
data_path = resolve_dataset_root("auto")
os.environ["DATA_PATH"] = data_path
print(f"Detected and verified TartanAir dataset at: {data_path}")


In [ ]:
# [5] Launch Dioptra-DINO Training (15 Epochs on Dual T4)
# Backbone LR = 2e-5 (fine-tuning DINOv2 visual features)
# Geometric Head LR = 2e-4 (learning scale & ARA attention bias)
# Batch size = 8 per GPU, 4-step gradient accumulation (effective batch size = 32)
!python /kaggle/working/dioptra_dino.py \
    --train auto \
    --epochs 15 \
    --batch-size 8 \
    --lr-backbone 2e-5 \
    --lr-head 2e-4


In [ ]:
# [5] Launch Dioptra-DINO Training (with 3D Virtual Normal Loss & Wide Pinhole Multiplier)
# --weight-normal 0.25 (3D Virtual Normal Loss for planar geometry & sharp boundaries)
# --crop-min 0.35 (Wide optical zoom simulation for infinite focal lengths)
# --batch-size 8 per GPU (effective batch size = 32 with 4-step accumulation)
# (Optional: add --image-size 336 --batch-size 4 for ultra-high-res 24x24 tokens)
!python /kaggle/working/dioptra_dino.py \
    --train "" \
    --epochs 15 \
    --batch-size 8 \
    --weight-normal 0.25 \
    --crop-min 0.35 \
    --lr-backbone 2e-5 \
    --lr-head 2e-4


In [ ]:
# [6] Verify output checkpoints
import os
print("Checkpoints in /kaggle/working/outputs_dino:")
for f in sorted(os.listdir('/kaggle/working/outputs_dino')):
    p = os.path.join('/kaggle/working/outputs_dino', f)
    size_mb = os.path.getsize(p) / (1024 * 1024)
    print(f"  {f} ({size_mb:.2f} MB)")

In [ ]:
# [7] Evaluate Dioptra-DINO, Generate Multi-FOV Sweep & Download Results
import os, sys, glob
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display, FileLink

# Ensure paths are ready
for p in ['/kaggle/working', '/kaggle/working/dioptra_repo']:
    if p not in sys.path and os.path.exists(p):
        sys.path.insert(0, p)

# 1. Verify Checkpoint
ckpt_path = '/kaggle/working/outputs_dino/dioptra_dino_best.pt'
if not os.path.exists(ckpt_path):
    candidates = sorted(glob.glob('/kaggle/working/outputs_dino/*.pt'))
    if candidates:
        ckpt_path = candidates[-1]
print(f'Using checkpoint: {ckpt_path} ({os.path.getsize(ckpt_path)/(1024*1024):.2f} MB)')

# 2. Import evaluation suite
try:
    from scripts.eval_dino import load_model, run_fov_sweep, run_benchmark, render_multiscene_grid
except ImportError:
    # Auto-clone/pull if repo not present
    if not os.path.exists('/kaggle/working/dioptra_repo'):
        os.system('git clone https://github.com/SeranomTheGreat/dioptra.git /kaggle/working/dioptra_repo')
    sys.path.insert(0, '/kaggle/working/dioptra_repo')
    from scripts.eval_dino import load_model, run_fov_sweep, run_benchmark, render_multiscene_grid

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_model(ckpt_path, device=device)

# 3. Discover validation / test samples in TartanAir dataset
sample_pairs = []
# Check TartanAir mounted directories
for p in sorted(glob.glob('/kaggle/input/**/image_left/*.png', recursive=True)):
    stem = os.path.splitext(os.path.basename(p))[0]
    cand_gts = [
        p.replace('image_left', 'depth_left').replace('.png', '.npy'),
        p.replace('image_left', 'depth_left').replace('.png', '_depth.npy'),
        os.path.join(os.path.dirname(p).replace('image_left', 'depth_left'), f'{stem}_left_depth.npy')
    ]
    for gt in cand_gts:
        if os.path.exists(gt):
            sample_pairs.append((p, gt))
            break
    if len(sample_pairs) >= 50:
        break

# Fallback: check test_samples from repo
if not sample_pairs:
    for p in sorted(glob.glob('/kaggle/working/**/test_samples/**/*_left.png', recursive=True)):
        base = p.replace('.png', '')
        if os.path.exists(f'{base}_depth.npy'):
            sample_pairs.append((p, f'{base}_depth.npy'))

print(f'Found {len(sample_pairs)} validation sample pairs for evaluation.')

# 4. Generate Multi-FOV Sweep (Figure 8 Style)
sweep_path = '/kaggle/working/outputs_dino/fig_dino_multi_fov_sweep.png'
if sample_pairs:
    test_img, test_gt = sample_pairs[0]
    print(f'Rendering Multi-FOV Sweep for: {test_img}')
    run_fov_sweep(model, test_img, test_gt, output_path=sweep_path, device=device)
    if os.path.exists(sweep_path):
        print('
=== MULTI-FOV SWEEP VISUALIZATION ===')
        display(IPImage(filename=sweep_path, width=950))

# 5. Multi-Scene Evaluation Grid
grid_path = '/kaggle/working/outputs_dino/fig_dino_multiscene_eval.png'
if len(sample_pairs) >= 2:
    print('Rendering Multi-Scene Evaluation Grid...')
    render_multiscene_grid(model, sample_pairs[:4], output_path=grid_path, device=device)
    if os.path.exists(grid_path):
        print('
=== MULTI-SCENE EVALUATION GRID ===')
        display(IPImage(filename=grid_path, width=950))

# 6. Quantitative Benchmark
if sample_pairs:
    print('
Computing held-out quantitative benchmark metrics...')
    from scripts.eval_dino import compute_metrics, preprocess_sample
    m_list = []
    for img_p, gt_p in sample_pairs[:30]:
        inp, gt_np, _, K_nat, _ = preprocess_sample(img_p, gt_p, img_size=224, device=device)
        if gt_np is not None:
            with torch.no_grad():
                pred = model(inp, K_nat).squeeze().cpu().numpy()
            m_list.append(compute_metrics(pred, gt_np))
    if m_list:
        print(f'=== HELD-OUT QUANTITATIVE RESULTS ({len(m_list)} samples) ===')
        print(f'AbsRel (Error ↓) : {np.mean([m["abs_rel"] for m in m_list]):.4f}')
        print(f'RMSE (m ↓)      : {np.mean([m["rmse"] for m in m_list]):.3f} m')
        print(f'δ < 1.25 (Acc ↑) : {np.mean([m["a1"] for m in m_list])*100:.2f}%')
        print(f'δ < 1.25² (Acc ↑): {np.mean([m["a2"] for m in m_list])*100:.2f}%')
        print(f'Scale Ratio      : {np.mean([m["scale_ratio"] for m in m_list]):.4f}')

# 7. Package Outputs for 1-Click Download
os.system('cd /kaggle/working && zip -q -r dioptra_dino_results.zip outputs_dino/')
print('
' + '='*70)
print('>>> SUCCESS! All DINO evaluations, sweeps, and models packaged! <<<')
print('Download archive: /kaggle/working/dioptra_dino_results.zip')
print('='*70)
display(FileLink('dioptra_dino_results.zip'))
